# Lezione 6 — Compressione video: movimento, bitrate e qualità
**Laboratorio di Trattamento dei Dati Multimediali · 90 minuti**

Perché un video può occupare molto meno della sequenza dei suoi fotogrammi non compressi? In questa lezione osserviamo la predizione temporale e sperimentiamo una codifica H.264/AVC.

## Obiettivi e prerequisiti
Al termine saprai interpretare un residuo di predizione, spiegare un vettore di movimento e confrontare bitrate, PSNR e artefatti visivi. Servono array NumPy, visualizzazioni con Matplotlib e i concetti di fotogramma, frame rate e compressione con perdita.

Collegamenti: video digitale (modulo 2), rateo e distorsione (modulo 5), predizione del movimento e H.264 (moduli 7–8). Il laboratorio riprende la lettura dei frame della lezione 4.

| Minuti | Attività |
|---|---|
| 0–10 | Setup e osservazione delle clip |
| 10–30 | Differenze, ricerca di un blocco e residui |
| 30–60 | H.264: tre impostazioni di qualità |
| 60–80 | Grafici e confronto visivo |
| 80–90 | Discussione dei risultati |

Le funzioni tecniche sono fornite in `lab_video.py`. Gli esercizi richiedono di cambiare parametri, formulare previsioni e interpretare risultati.

## 0. Setup — prima della lezione
Apri questo notebook con un kernel Python 3. Installa le dipendenze una sola volta, se necessario, eseguendo la cella commentata sotto. Per ricodificare servono FFmpeg nel PATH e l'encoder `libx264`.

**Percorso senza FFmpeg:** imposta `RICODIFICA = False`. Le sei codifiche, le misure e alcuni fotogrammi decodificati sono già inclusi in `materiali`; le altre attività funzionano con NumPy e Matplotlib. Non serve una connessione durante la lezione.

In [ ]:
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path
import sys, csv, shutil, time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

BASE = Path.cwd()
if not (BASE / 'lab_video.py').exists():
    BASE = BASE / '006'
if not (BASE / 'lab_video.py').exists():
    raise RuntimeError('Apri il notebook dalla cartella 006 o dalla radice LAB-TDM.')
sys.path.insert(0, str(BASE))
from lab_video import (genera_clip, grigio, cerca_blocco, codifica, decodifica,
                       misura, WIDTH, HEIGHT, FPS, N_FRAMES, CRFS, NAMES)
plt.rcParams.update({'figure.figsize': (10, 4), 'image.cmap': 'gray'})
MATERIALI = BASE / 'materiali'
RICODIFICA = False  # True per eseguire le sei codifiche durante il laboratorio
print('FFmpeg:', shutil.which('ffmpeg') or 'assente: usa i materiali forniti')

## 1. Due video, un esperimento controllato
Generiamo due clip RGB di 5 secondi, 320×180 pixel, 24 fps. Sfondo, oggetto e traiettoria coincidono; cambia la velocità: 1 o 6 pixel per fotogramma. La traiettoria torna periodicamente al punto iniziale: nella clip veloce osserverai anche rientri bruschi, che contribuiscono alla difficoltà di predizione.

Il riferimento è l'array RGB generato, prima della compressione. Le anteprime qui sotto sono già compresse a CRF 18 e servono solo a osservare il movimento.

**Prima di misurare:** quale clip richiederà più dati a parità di impostazione del codec? Formula un'ipotesi, senza considerarla una legge valida per tutti i video.

In [ ]:
clips = {'lento': genera_clip(1), 'veloce': genera_clip(6)}
for nome in NAMES:
    print(nome)
    display(Video(filename=str(MATERIALI / f'{nome}_crf18.mp4'), embed=True, width=480))
raw_bytes = WIDTH * HEIGHT * 3 * N_FRAMES
print(f'RGB non compresso: {raw_bytes/1e6:.2f} MB per clip (MB decimali)')

## 2. Predizione temporale — 20 minuti
### 2.1 Usare il fotogramma precedente senza spostamenti
Un primo predittore del frame corrente è il precedente. Il residuo è **corrente − predizione**. Dove l'immagine non cambia, il residuo è nullo; lungo l'oggetto in movimento compaiono differenze.

La sottrazione va eseguita in virgola mobile: con `uint8` i valori negativi non sono rappresentabili. Per la dimostrazione lavoriamo su una combinazione pesata dei canali RGB, una luma approssimata.

In [ ]:
nome = 'veloce'
t = 8
precedente, corrente = clips[nome][t-1:t+1]
residuo = grigio(corrente) - grigio(precedente)
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].imshow(precedente); ax[0].set_title('Precedente')
ax[1].imshow(corrente); ax[1].set_title('Corrente')
im = ax[2].imshow(residuo, cmap='RdBu_r', vmin=-255, vmax=255)
ax[2].set_title('Residuo senza movimento')
for a in ax: a.axis('off')
fig.colorbar(im, ax=ax[2], shrink=.7)
plt.show()

### 2.2 Cercare un blocco nel frame precedente
Per un blocco 16×16 del frame corrente cerchiamo, entro ±12 pixel, il blocco del riferimento che minimizza la **SAD**, somma delle differenze assolute. Il vettore restituito punta **dal blocco corrente alla posizione del predittore nel precedente**: per un oggetto che si sposta a destra, ci aspettiamo una componente orizzontale negativa.

Questo è un modello didattico: H.264 usa anche altre dimensioni di blocco, predizione frazionaria e scelte che tengono conto del costo in bit. La nostra ricerca non ricostruisce i vettori scelti da H.264.

In [ ]:
# Coordinate valide per il frame t=8 della clip veloce.
x, y, lato = 64, 70, 16
vettore, senza, compensato = cerca_blocco(precedente, corrente, x, y, lato, raggio=12)
print('Vettore verso il riferimento (dx, dy):', vettore)
print(f'MAE senza compensazione: {np.abs(senza).mean():.2f}')
print(f'MAE con compensazione:   {np.abs(compensato).mean():.2f}')
fig, ax = plt.subplots(1, 3, figsize=(11, 3))
ax[0].imshow(corrente)
ax[0].add_patch(plt.Rectangle((x,y), lato,lato, fill=False, edgecolor='yellow'))
ax[0].arrow(x+lato/2, y+lato/2, *vettore, color='red', head_width=4)
ax[0].set_title('Blocco corrente e vettore')
for a, r, titolo in zip(ax[1:], [senza, compensato], ['Residuo senza compensazione', 'Residuo compensato']):
    a.imshow(r, cmap='RdBu_r', vmin=-255, vmax=255); a.set_title(titolo)
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

**Esercizio guidato (5 minuti).** Ripeti la ricerca con `raggio=2`, poi con `raggio=12`. Prima prevedi l'effetto, poi confronta vettore ed errore. È sufficiente cercare più lontano per risolvere qualsiasi cambiamento?

Un residuo più piccolo può favorire la compressione, ma la sua MAE non è una misura dei bit necessari. Occorre trasmettere anche le informazioni di predizione; contano la struttura del residuo e le altre fasi di codifica. Le aree appena scoperte e i cambi di scena possono essere difficili da predire.

## 3. Codifica H.264 — 30 minuti
### Dal predittore al file
Un codec combina predizione, trasformazione del residuo, quantizzazione e codifica entropica. La quantizzazione introduce perdita. H.264 può usare predizione interna al frame (intra) e fra frame (inter).

Confrontiamo **CRF 18, 28 e 38**, lasciando invariati preset `fast`, formato `yuv420p` e intervallo massimo tra keyframe di 48 frame. CRF più alto richiede generalmente meno bit con maggiore perdita; **CRF non è un bitrate e non garantisce identica qualità percettiva su scene diverse**. Non cambiamo contemporaneamente il preset.

`yuv420p` include sottocampionamento cromatico. Il confronto con il riferimento RGB misura anche l'effetto della conversione di colore, non solo la quantizzazione. Queste opzioni sono visibili nella funzione `codifica` del file di supporto.

**Attività:** prevedi l'ordine di dimensioni e qualità. Con FFmpeg disponibile, imposta `RICODIFICA=True` nel setup e riesegui le celle successive. Altrimenti analizza i risultati forniti: derivano dallo stesso esperimento.

In [ ]:
righe = []
campioni = {}
INDICI = np.array([8, 40, 80])
inizio = time.perf_counter()
if RICODIFICA:
    CARTELLA_VIDEO = BASE / 'risultati'
    for nome, originale in clips.items():
        for crf in CRFS:
            path = CARTELLA_VIDEO / f'{nome}_crf{crf}.mp4'
            codifica(originale, path, crf)
            ricostruito = decodifica(path)
            righe.append(dict(clip=nome, crf=crf, **misura(originale, ricostruito, path)))
            campioni[f'{nome}_{crf}'] = ricostruito[INDICI].copy()
            print('Completato:', path.name)
else:
    CARTELLA_VIDEO = MATERIALI
    with (MATERIALI / 'misure.csv').open() as f:
        for r in csv.DictReader(f):
            righe.append(dict(clip=r['clip'], crf=int(r['crf']), byte=int(r['byte']),
                **{k: float(r[k]) for k in ['bitrate_kbps','mse_rgb','psnr_rgb_db']}))
    with np.load(MATERIALI / 'campioni.npz') as archivio:
        campioni = {k: archivio[k].copy() for k in archivio.files}
print(f'Tempo di questa cella: {time.perf_counter()-inizio:.1f} s')

### Che cosa misuriamo?
- **Bitrate medio del file** = byte × 8 / durata. Include l'overhead MP4; i file non contengono audio. Non è il solo bitrate del flusso elementare.
- **MSE RGB** = media del quadrato dell'errore su tutti i frame, pixel e canali.
- **PSNR RGB** = 10 log₁₀(255²/MSE), in dB. Più alto indica minore errore quadratico; non implica necessariamente migliore percezione. Qui calcoliamo il PSNR dal MSE globale, senza mediare i PSNR dei frame.

Il riferimento e il decodificato devono avere stessi frame, dimensioni e allineamento temporale. Il rapporto fra RGB grezzo e MP4 include anche la differenza di rappresentazione cromatica.

In [ ]:
print(f"{'Clip':8} {'CRF':>4} {'kB':>9} {'kb/s':>9} {'PSNR dB':>9} {'RGB/MP4':>9}")
for r in righe:
    print(f"{r['clip']:8} {r['crf']:4d} {r['byte']/1000:9.2f} "
          f"{r['bitrate_kbps']:9.2f} {r['psnr_rgb_db']:9.2f} {raw_bytes/r['byte']:9.1f}")

## 4. Bitrate e qualità — 20 minuti
### 4.1 Leggere le curve
Ogni punto rappresenta una codifica. Colleghiamo i punti solo per facilitarne la lettura: tre campioni non descrivono l'intera curva rateo–distorsione.

Confronta prima i tre punti **della stessa clip**. Il confronto fra clip diverse richiede cautela: cambiano i contenuti temporali e lo stesso PSNR non equivale necessariamente alla stessa qualità percepita.

In [ ]:
fig, ax = plt.subplots()
for nome in NAMES:
    dati = sorted([r for r in righe if r['clip']==nome], key=lambda r:r['bitrate_kbps'])
    ax.plot([r['bitrate_kbps'] for r in dati], [r['psnr_rgb_db'] for r in dati], 'o-', label=nome)
    for r in dati:
        ax.annotate(f"CRF {r['crf']}", (r['bitrate_kbps'],r['psnr_rgb_db']), xytext=(4,5), textcoords='offset points')
ax.set(xlabel='Bitrate medio del file (kb/s)', ylabel='PSNR RGB globale (dB)', title='H.264: bitrate e qualità sulle due clip')
ax.margins(x=.12, y=.15)
ax.grid(alpha=.3); ax.legend(); plt.tight_layout()
(BASE / 'risultati').mkdir(exist_ok=True)
fig.savefig(BASE / 'risultati' / 'bitrate_qualita.png', dpi=150)
plt.show()

### 4.2 Guardare gli artefatti
Osserva un ritaglio del riferimento e delle tre ricostruzioni con la stessa scala e senza interpolazione. Cerca alterazioni dei contorni, perdita di dettaglio e cambiamenti cromatici. Gli artefatti non devono necessariamente essere tutti presenti.

Modifica `indice_campione` tra 0, 1 e 2, e `nome` tra `lento` e `veloce`. Poi guarda i video: un singolo frame non descrive gli artefatti nel tempo.

In [ ]:
nome = 'veloce'
indice_campione = 1
t = int(INDICI[indice_campione])
velocita = 1 if nome == 'lento' else 6
pos = 16 + (velocita*t) % 240
ritaglio = (slice(58,122), slice(pos-8,pos+56))
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
ax[0].imshow(clips[nome][t][ritaglio], interpolation='nearest'); ax[0].set_title('Riferimento RGB')
for a, crf in zip(ax[1:], CRFS):
    a.imshow(campioni[f'{nome}_{crf}'][indice_campione][ritaglio], interpolation='nearest')
    a.set_title(f'CRF {crf}')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
for crf in (18,38):
    print(f'{nome}, CRF {crf}')
    display(Video(filename=str(CARTELLA_VIDEO / f'{nome}_crf{crf}.mp4'), embed=True, width=480))